# 06 · Representations (Table 7) and steering (Table 9)

**Runtime: L4 or A100** (a T4 works, slower). Needs the generations (notebook 01), the LaBSE embeddings (notebook 01's free-text cell) and both adapters (notebook 04). One model load serves both tables: Qwen in 4-bit with the en_aligned and multi_aligned adapters, switched per checkpoint (as in the manipulation check).

| Step | What | Output on Drive (`MyDrive/mpac_hi_pt/`) | A100 |
|---|---|---|---|
| Table 7 | Last-token activations of every eval prompt, per checkpoint and language; cross-lingual similarity per layer, its link to behavioural divergence, and how much of the English DPO direction shows up in hi / pt | `activations/<model>/<checkpoint>/<lang>.npz` (~100 MB each), `results/<model>/interp/table7.json` | ~5–10 min |
| Table 9 | Base model, forced-choice items: add a direction to the residual stream at one layer and re-measure ACS and TruthfulQA accuracy. Directions from **tune** items, scores on **test** items | `results/<model>/interp/table9_<lang>_<objective>.json` | ~5 min per objective |

How to read them: `src/mpac/interp/activations.py` (module docstring). Table 9 is the causal check: a drop in ACS for `to_english` or `alignment` that `random` (same norm) doesn't give.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[gpu,signals]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
MODEL = "qwen2.5-7b"
CONFIGS = ["configs/hi_pt.yaml"]

import torch
from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.config import get_model
from mpac.paths import resolve_root
cfg = load_config(*CONFIGS)
m = get_model(cfg, MODEL)
print("model:", m["id"], "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| artifacts ->", resolve_root(cfg))

In [ ]:
# Load once: base in 4-bit + both LoRA adapters (training/heldout.load_scoring_model)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from mpac.training.heldout import load_scoring_model
tok, model = load_scoring_model(cfg, m)

## Table 7 · Representation similarity

In [ ]:
import pandas as pd
from mpac.interp.activations import run_table7

t7 = run_table7(cfg, m, tok, model)
print("saved ->", t7["path"], "\n")
summ = pd.DataFrame(t7["summary"])
sim = summ[summ.get("kind").isna()] if "kind" in summ else summ
print("Cross-lingual similarity (mean-centred cosine; gap = matched - mismatched; mid = middle layer)")
print(sim.drop(columns=["per_objective", "kind"], errors="ignore").round(3).to_string(index=False), "\n")
print("Per objective, gap at the middle layer")
print(pd.DataFrame({f"{r['checkpoint']}/{r['lang']}": {o: v["gap_mid"] for o, v in r["per_objective"].items()}
                    for _, r in sim.iterrows()}).round(3).to_string(), "\n")
if "kind" in summ:
    print("English DPO direction in the target language (transfer 1 = all of it)")
    print(summ[summ["kind"].eq("dpo_direction")][["checkpoint", "lang", "cos_mid", "transfer_mid", "transfer_mean"]]
          .round(3).to_string(index=False))

In [ ]:
# Per-layer curves: gap (base), behaviour link (Spearman of item similarity vs divergence), DPO transfer
gap = pd.DataFrame({k: v["gap"] for k, v in t7["similarity"].items()})
link = pd.DataFrame(t7["behaviour_link"]).add_prefix("rho ")
tr = pd.DataFrame({k: v["transfer"] for k, v in t7["transfer"].items()}).add_prefix("transfer ")
print(pd.concat([gap, link, tr], axis=1).round(3).to_string())

## Table 9 · Steering (causal check)

Runs on the Hindi cells first (where collapse is largest), then Portuguese as the contrast. `alpha` is in units of the language shift at that layer (`shift_norm`), so 1.0 moves the Hindi mean all the way to the English one. `acs_diff` < 0 = less collapse than unsteered, with a 95% paired bootstrap CI and permutation p; `acc_diff` = TruthfulQA accuracy change (honesty only).

In [ ]:
from mpac.interp.activations import run_table9

t9 = {}
for lang in ["hi", "pt"]:
    for obj in ["honesty", "pluralistic_value"]:
        r = run_table9(cfg, m, lang, [obj], tok=tok, model=model)
        t9[(lang, obj)] = r
        df = pd.DataFrame(r["rows"])
        print(f"== {lang} / {obj}: {r['n_test']} test items (directions from {r['n_tune']} tune items) -> {r['path']}")
        cols = [c for c in ["direction", "layer", "alpha", "acs", "acs_diff", "acs_ci", "acs_p", "acc", "acc_diff", "acc_p"] if c in df]
        print(df[cols].round(3).to_string(index=False), "\n")